In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.bankofbaroda.in/interest-rate-and-service-charges/deposits-interest-rates" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find_all("table")
    table1 = table[0]
    table2 = table[1]
    rows=table1.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[1:]:  #Skip heading
        data = i.find_all("td")
        row=['Bank of Baroda']
        row.extend([tr.text for tr in data[:3]])
        row.insert(3,'')
        row.insert(5,'')
        for i in range(0,5):
            row[i] = ' '.join(row[i].strip().replace('\n', '').split())
        if row_count == 11:
            tax_saver_row = ['Bank of Baroda', 'Tax-saver Fixed Deposit', row[2], '', row[4], '']
            list_of_interest_rates.append(tax_saver_row)
        list_of_interest_rates.append(row)
        row_count += 1
    
    rows=table2.find_all("tr")
    row_count = 0
    for i in rows[1:]:  #Skip heading
        data = i.find_all("td")
        row=['Bank of Baroda']
        row.extend([tr.text for tr in data[:3]])
        row.insert(3,'')
        row.insert(5,'')
        #row.extend(['',str(30000000)])
        for i in range(0,5):
            row[i] = ' '.join(row[i].strip().replace('\n', '').split())
        #print("\nrow = ",row)
        list_of_interest_rates.append(row)
    
    list_of_interest_rates = [[str(item).strip('*$#') for item in row] for row in list_of_interest_rates]
    for list in list_of_interest_rates: 
        for i in range(2,5,2): list[i] = str(list[i].strip()) + '%'
    #print(list_of_interest_rates)
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nBank of Baroda = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 21112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_21112024.csv
         bank_name                                      tenure gen_rate  \
0   Bank of Baroda                           7 days to 14 days    4.25%   
1   Bank of Baroda                          15 days to 45 days    4.50%   
2   Bank of Baroda                          46 days to 90 days    5.50%   
3   Bank of Baroda                         91 days to 180 days    5.60%   
4   Bank of Baroda                        181 days to 210 days    5.75%   
5   Bank of Baroda                        211 days to 270 days    6.25%   
6   Bank of Baroda       271 days & above and less than 1 year    6.50%   
7   Bank of Baroda                                      1 year    6.85%   
8   Bank of Baroda  Above 1 year to 400 days (except 400 Days)    7.00%   
9   Bank of Baroda             Above 400 days and upto 2 Years    7.00%   
10  Bank of Baroda              Above 2 Years and upto 3 Years    7.15%   